# 10  All saved LOSO experiments: Shapley interpretation (output deleted due to upload problems)

This version discovers **all saved experiments within one artifact root**: primary models (including logistic regression and majority), binary secondary tasks, availability analyses, RF/XGBoost/LightGBM/MLP, late fusion, modality-dropout scenarios, four-class models and notebook09 sensitivity variants when saved with provenance.

No SHAP/Numba import, model fitting or feature selection. Model-agnostic Monte Carlo Shapley estimates explain **probabilities for every algorithm**, not XGBoost margins. Do not combine these outputs with the older native-margin explanations. Explanations use only held-out windows; reference rows are sampled from training participants. Correlated-feature hybrids may be unrealistic; no causal interpretation.

The saved calibration/uncertainty workflow does not persist all fitted estimators/calibrators. Its predictions alone cannot reconstruct a prediction function, so it is audited as unavailable rather than assigned invented attributions. Synthetic and historical artifact roots are not mixed.

In [1]:
from pathlib import Path
import os,sys,json,ast,re,itertools,warnings,hashlib
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator,TransformerMixin
import joblib
from IPython.display import display,Image,Markdown

ARTIFACTS_REL='artifacts/revision_v2'
PROJECT_ROOT=None
here=Path.cwd().resolve()
ROOT=Path(PROJECT_ROOT).resolve() if PROJECT_ROOT else next(
    (p for p in [here,*here.parents] if (p/'configs/config.yaml').is_file() and (p/'src').is_dir()),None)
if ROOT is None:raise RuntimeError('Open this notebook in the project notebooks directory.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.fusion import comparison_tables
from src.exploratory_fusion import mask_modality
A=ROOT/ARTIFACTS_REL
PAIRS=32;BACKGROUND_PER_SUBJECT=10;MAX_BACKGROUND=150;BATCH_SIZE=4096;SEED=42
SHOW_FIGURES=True;TOP_N=15;BEESWARM_WINDOWS_PER_SUBJECT=20
WATERFALL_SUBJECTS=[2];WATERFALL_WINDOW_ID=None
LOW_CORRELATION=.30;HIGH_CORRELATION=.70;EFFECT_TOP_N=3;ALE_BINS=6;PDP_POINTS=15
RUN_EFFECT_PLOTS=True
OUT=A/'interpretability'/f'all_loso_probability_pairs_{PAIRS}'
OUT.mkdir(parents=True,exist_ok=True)
CLASS_NAMES={0:'Non-stress',1:'Baseline',2:'Stress',3:'Amusement',4:'Meditation'}
def class_name(task,k):
    return {0:'Non-stress',1:'Stress'}[int(k)] if task=='stress_vs_nonstress' else CLASS_NAMES.get(int(k),str(k))
def safe(s):return re.sub(r'[^a-zA-Z0-9_.-]+','_',str(s))
def pretty(s):return s.replace('__',' · ').replace('_',' ')
def show(fig,path):
    path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=300,bbox_inches='tight')
    if SHOW_FIGURES:
        with BytesIO() as b:
            fig.savefig(b,format='png',dpi=90,bbox_inches='tight');image=b.getvalue()
        display(Image(data=image))
    plt.close(fig)

# Restore only the trusted selector definition; do not execute notebook09 training.
for name in ['09_feature_redundancy_and_tree_importance_v2.ipynb','09_feature_redundancy_and_tree_importance.ipynb']:
    nbpath=ROOT/'notebooks'/name
    if not nbpath.is_file():continue
    book=json.loads(nbpath.read_text(encoding='utf-8'))
    for cell in book['cells']:
        if cell['cell_type']=='code' and 'class CorrelationSelector' in ''.join(cell['source']):
            parsed=ast.parse(''.join(cell['source']))
            node=next(n for n in parsed.body if isinstance(n,ast.ClassDef) and n.name=='CorrelationSelector')
            exec(compile(ast.Module(body=[node],type_ignores=[]),'<saved selector>','exec'),globals())
            break
    if 'CorrelationSelector' in globals():break
print('All-experiment interpretation configured.')


All-experiment interpretation configured.


## Automatic discovery and provenance

Every result JSON with task, model, site, classes and folds is paired to its OOF CSV and model directory. Full primary, secondary and exploratory experiments are independent entries. Reduced notebook09 models need sidecar metadata written during fitting. Missing files are listed explicitly. Every loaded predictor must reproduce its saved OOF probabilities before any interpretation is retained.

The directory inventory is independent of which predictive algorithm exposes `feature_importances_`: this method also explains linear models, MLP and composite fusion predictors. Majority models should yield zero contributions, apart from floating-point error.

In [2]:
registry=[];missing=[]
result_roots=[A/'results',A/'exploratory/results',A/'exploratory/four_class/results']
for result_root in result_roots:
    for path in sorted(result_root.glob('*.json')):
        result=json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(result,dict) or not all(k in result for k in ['task','model','site','classes','folds']):continue
        experiment=path.stem;parent=result_root.parent
        oof=path.with_name(path.stem+'_predictions.csv')
        method=result.get('method');scenario=result.get('scenario','both_present')
        for fold in result['folds']:
            sid=int(fold['subject_id'])
            folder=parent/'models'/(method if method in ['late_fusion','modality_dropout'] else f"{result['task']}_{result['site']}_{result['model']}")
            model_file=folder/f'S{sid}.joblib';meta=folder/f'S{sid}_metadata.json'
            absent=[]
            if not model_file.is_file():absent.append('model')
            if not oof.is_file():absent.append('OOF')
            if not method and not meta.is_file():absent.append('metadata')
            row=dict(experiment=experiment,task=result['task'],site=result['site'],model=result['model'],variant='full',
                subject_id=sid,classes=result['classes'],method=method,scenario=scenario,model_file=model_file,
                metadata_file=meta,prediction_file=oof,feature_root=parent/'features' if result['task']=='four_class' else A/'features')
            if absent:missing.append({**{k:row[k] for k in ['experiment','task','site','model','variant','subject_id']},'reason':'missing '+', '.join(absent)})
            else:registry.append(row)

filter_root=A/'exploratory/feature_redundancy_saved_models'
if not filter_root.is_dir():filter_root=A/'exploratory/feature_redundancy_importance'
metrics_path=filter_root/'fold_metrics.csv'
if metrics_path.is_file():
    metric=pd.read_csv(metrics_path)
    for r in metric[['site','model','variant','subject_id']].drop_duplicates().itertuples(index=False):
        sid=int(r.subject_id);stem=f'{r.site}_{r.model}_{r.variant}_S{sid}'
        model_file=filter_root/f'{stem}.joblib';meta=filter_root/f'{stem}_metadata.json';oof=filter_root/'predictions.csv'
        absent=[name for name,p in [('model',model_file),('metadata',meta),('OOF',oof)] if not p.is_file()]
        experiment=f'sensitivity_{r.site}_{r.model}_{r.variant}'
        if absent:missing.append(dict(experiment=experiment,task='three_class',site=r.site,model=r.model,variant=r.variant,subject_id=sid,reason='missing '+', '.join(absent)))
        else:registry.append(dict(experiment=experiment,task='three_class',site=r.site,model=r.model,variant=r.variant,
            subject_id=sid,classes=[1,2,3],method=None,scenario='both_present',model_file=model_file,
            metadata_file=meta,prediction_file=oof,feature_root=A/'features'))
if (A/'results/uncertainty_predictions.csv').is_file():
    missing.append(dict(experiment='uncertainty_calibrated',reason='fitted estimators/calibrators not persisted by uncertainty workflow'))
registry=pd.DataFrame(registry)
pd.DataFrame(missing).to_csv(OUT/'unavailable_experiments.csv',index=False)
if registry.empty:raise RuntimeError('No complete saved experiments found. Check ARTIFACTS_REL.')
display(registry.groupby(['task','site','model','variant']).size().rename('available_folds').reset_index())
display(pd.DataFrame(missing))
REQUIRE_COMPLETE_REDUCED=True
if REQUIRE_COMPLETE_REDUCED:
    expected={(site,model,variant,sid)
        for site in ['chest','wrist','early_fusion']
        for model in ['logistic_regression','random_forest','xgboost','lightgbm','mlp']
        for variant in ['reduced_0.95','reduced_0.90','reduced_0.80']
        for sid in sorted(getattr(registry[registry.site==site],'subject_id').unique())}
    actual={(r.site,r.model,r.variant,r.subject_id) for r in registry.itertuples()
            if r.experiment.startswith('sensitivity_')}
    absent=expected-actual
    if absent:
        display(pd.DataFrame(sorted(absent),columns=['site','model','variant','subject_id']))
        raise RuntimeError('Reduced-model coverage incomplete. Run notebook09b to save pipelines, metadata and OOF predictions, then rerun this registry. No SHAP calculation was started.')
registry.drop(columns=['model_file','metadata_file','prediction_file','feature_root']).to_csv(OUT/'model_registry.csv',index=False)


,task,site,model,variant,available_folds
0,availability_three_class,chest,logistic_regression,full,15
1,availability_three_class,wrist,logistic_regression,full,15
2,baseline_vs_stress,chest,logistic_regression,full,15
3,baseline_vs_stress,early_fusion,logistic_regression,full,15
4,baseline_vs_stress,wrist,logistic_regression,full,15
...,...,...,...,...,...
79,three_class,wrist,xgboost,reduced_0.95,15
80,three_class_both_present,late_fusion,logistic_regression,full,15
81,three_class_both_present,modality_dropout,logistic_regression,full,15
82,three_class_chest_missing,modality_dropout,logistic_regression,full,15


,experiment,reason
0,uncertainty_calibrated,fitted estimators/calibrators not persisted by...


In [3]:
frame_cache={}
def get_frames(root):
    key=str(root)
    if key not in frame_cache:
        tables={s:pd.read_parquet(root/f'{s}_features.parquet') for s in ['chest','wrist']}
        for frame in tables.values():
            if frame.valid.dtype!=bool:
                text=frame.valid.astype(str).str.lower()
                if not text.isin(['true','false']).all():raise ValueError('Invalid validity flags.')
                frame['valid']=text.eq('true')
        paired=comparison_tables(tables,True);available=comparison_tables(tables,False)
        frame_cache[key]=(paired,available)
    return frame_cache[key]

def prepare(r):
    paired,available=get_frames(r.feature_root)
    frame=(available[r.site] if r.task=='availability_three_class' else
           paired['early_fusion'] if r.method in ['late_fusion','modality_dropout'] else paired[r.site]).copy()
    if r.task=='stress_vs_nonstress':frame['label']=frame.label.eq(2).astype(int)
    elif r.task=='baseline_vs_stress':frame=frame[frame.label.isin([1,2])].copy()
    else:frame=frame[frame.label.isin(r.classes)].copy()
    artifact=joblib.load(r.model_file)
    if r.method in ['late_fusion','modality_dropout']:
        fit_subjects=list(map(int,artifact['fit_subjects']))
        bundle=artifact['bundle']
        if r.method=='late_fusion':
            cols=sorted(set().union(*(set(bundle[s]['features']) for s in ['chest','wrist'])))
            weights=artifact['metadata']['fixed_probability_weights']
            def predict(X):
                outputs=[]
                for s in ['chest','wrist']:
                    m=bundle[s]['model']
                    if list(map(int,m.classes_))!=list(r.classes):raise ValueError('Composite class order mismatch.')
                    outputs.append(weights[s]*m.predict_proba(X[bundle[s]['features']]))
                return sum(outputs)
        else:
            model=bundle['model'];stored=bundle['features']
            cols=[col for col in stored if '__available' not in col]
            if list(map(int,model.classes_))!=list(r.classes):raise ValueError('Composite class order mismatch.')
            def predict(X):return model.predict_proba(mask_modality(X,r.scenario)[stored])
    else:
        meta=json.loads(r.metadata_file.read_text(encoding='utf-8'))
        if int(meta['test_subject'])!=r.subject_id:raise ValueError('Metadata test participant mismatch.')
        fit_subjects=list(map(int,meta['fit_subjects']));cols=meta['features'];model=artifact
        if list(map(int,model.classes_))!=list(r.classes):raise ValueError('Class order mismatch.')
        def predict(X):return model.predict_proba(X[cols])
    if r.subject_id in fit_subjects or set(fit_subjects)!=set(frame.subject_id.unique())-{r.subject_id}:
        raise ValueError('Invalid LOSO train/test provenance.')
    train=frame[frame.subject_id.isin(fit_subjects)].copy()
    test=frame[frame.subject_id.eq(r.subject_id)].copy()
    oof=pd.read_csv(r.prediction_file)
    if r.experiment.startswith('sensitivity_'):
        oof=oof[(oof.site==r.site)&(oof.model==r.model)&(oof.variant==r.variant)]
    oof=oof[oof.subject_id.eq(r.subject_id)]
    if oof.window_id.duplicated().any() or set(oof.window_id)!=set(test.window_id):raise ValueError('OOF windows mismatch.')
    test=test.set_index('window_id').loc[oof.window_id].reset_index()
    if not np.array_equal(test.label.to_numpy(),oof.label.to_numpy()):raise ValueError('OOF labels mismatch.')
    expected=oof[[f'p_{k}' for k in r.classes]].to_numpy(float)
    if not np.allclose(predict(test[cols]),expected,rtol=1e-5,atol=1e-6):raise ValueError('Model does not reproduce saved OOF probabilities.')
    background=pd.concat([g.sample(min(len(g),BACKGROUND_PER_SUBJECT),random_state=SEED) for _,g in train.groupby('subject_id')])
    if len(background)>MAX_BACKGROUND:background=background.sample(MAX_BACKGROUND,random_state=SEED)
    return predict,train,test,cols,background[cols]

def shapley_sampling(predict,X,background,n_pairs=PAIRS,seed=SEED):
    if n_pairs<2:raise ValueError('At least two sampling pairs required.')
    rng=np.random.default_rng(seed);n,p=X.shape;actual=np.asarray(predict(X),float);k=actual.shape[1]
    mean=np.zeros((n,p,k));m2=np.zeros_like(mean);base_total=np.zeros(k)
    for iteration in range(n_pairs):
        reference=background.iloc[[rng.integers(len(background))]]
        base_total+=np.asarray(predict(reference),float)[0]
        permutation=rng.permutation(p);contribution=np.zeros_like(mean)
        for order in [permutation,permutation[::-1]]:
            state=pd.DataFrame(np.repeat(reference.to_numpy(copy=True),n,axis=0),columns=X.columns)
            states=[state.copy(deep=True)]
            for j in order:
                state.iloc[:,j]=X.iloc[:,j].to_numpy(copy=True);states.append(state.copy(deep=True))
            stack=pd.concat(states,ignore_index=True)
            output=np.concatenate([np.asarray(predict(stack.iloc[a:a+BATCH_SIZE]),float) for a in range(0,len(stack),BATCH_SIZE)])
            increments=np.diff(output.reshape(p+1,n,k),axis=0)
            for step,j in enumerate(order):contribution[:,j,:]+=.5*increments[step]
        delta=contribution-mean;mean+=delta/(iteration+1);m2+=delta*(contribution-mean)
    base=np.tile(base_total/n_pairs,(n,1));error=float(np.max(np.abs(base+mean.sum(1)-actual)))
    if not np.allclose(base+mean.sum(1),actual,atol=2e-5,rtol=2e-5):raise ValueError('Shapley additivity check failed.')
    se=np.sqrt(np.maximum(m2/(n_pairs-1),0)/n_pairs)
    return mean,base,se,error

def ale(predict,X,feature,edges):
    values=X[feature].to_numpy(float);bins=np.clip(np.searchsorted(edges,values,side='right')-1,0,len(edges)-2)
    eligible=np.isfinite(values)&(values>=edges[0])&(values<=edges[-1]);counts=[];increments=[]
    for i in range(len(edges)-1):
        rows=X.loc[eligible&(bins==i)].copy()
        if rows.empty:return None
        lo=rows.copy();hi=rows.copy();lo[feature]=edges[i];hi[feature]=edges[i+1]
        counts.append(len(rows));increments.append((predict(hi)-predict(lo)).mean(0))
    increments=np.asarray(increments);effects=np.cumsum(increments,axis=0)-increments/2
    effects-=np.average(effects,axis=0,weights=counts)
    return (edges[:-1]+edges[1:])/2,effects,counts

def family(feature):
    return {'ECG':'Cardiac ECG/HRV','BVP':'Pulse BVP/PRV','EDA':'Electrodermal','ACC':'Movement',
            'Temp':'Temperature','TEMP':'Temperature','Resp':'Respiration','EMG':'Muscle'}.get(feature.split('__')[1],'Other')


## Execute saved folds

All discovered experiments run by default. This can be expensive, particularly MLP and composite predictors. Each completed fold is checkpointed. Increasing sampling pairs creates a separate output root. Warnings are captured by category rather than displayed; raw exceptions and private paths are not printed. Status and errors remain visible as safe descriptions. No changes to source fingerprints or training artifacts are made.

The writable-copy correction is explicit: correlation matrices are copied before changing the diagonal. PDP is limited to low training correlation; ALE to high correlation, with training bin edges and held-out reference rows. No multidimensional group ALE is claimed.

In [ ]:
status=[];feature_rows=[];group_rows=[];diagnostics=[]
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('once')
    for r in registry.itertuples():
        slug=f'{r.experiment}_S{r.subject_id}';folder=OUT/'folds'/slug;folder.mkdir(parents=True,exist_ok=True)
        common=dict(experiment=r.experiment,task=r.task,site=r.site,model=r.model,variant=r.variant,subject_id=r.subject_id)
        try:
            predict,train,test,cols,bg=prepare(r)
            X=test[cols].copy()
            values,base,se,error=shapley_sampling(predict,X,bg,PAIRS,SEED+r.subject_id)
            pieces=[]
            for j,k in enumerate(r.classes):
                table=pd.DataFrame(values[:,:,j],columns=cols)
                table.insert(0,'window_id',test.window_id.to_numpy());table['class_label']=k;table['base_value']=base[:,j]
                table['predicted_probability']=base[:,j]+values[:,:,j].sum(1)
                for key,value in common.items():table[key]=value
                pieces.append(table)
            pd.concat(pieces,ignore_index=True).to_parquet(folder/f'shap_values_fold_{r.subject_id}.parquet',index=False)
            np.savez_compressed(folder/'monte_carlo_standard_errors.npz',standard_errors=se,features=np.asarray(cols),classes=np.asarray(r.classes))
            # Compute per-fold statistics locally so a plotting failure cannot contaminate aggregation.
            local_features=[];local_groups=[]
            for j,k in enumerate(r.classes):
                for i,feature in enumerate(cols):local_features.append({**common,'class_label':k,'feature':feature,
                    'mean_abs_shap':float(np.abs(values[:,i,j]).mean()),'mean_signed_shap':float(values[:,i,j].mean()),
                    'mean_mc_se':float(se[:,i,j].mean())})
                for grouping,mapper in [('family',family),('modality',lambda f:f.split('__')[0])]:
                    for name in sorted({mapper(f) for f in cols}):
                        indices=[i for i,f in enumerate(cols) if mapper(f)==name]
                        local_groups.append({**common,'class_label':k,'grouping':grouping,'group':name,
                            'mean_abs_group_sum':float(np.abs(values[:,indices,j].sum(1)).mean()),
                            'sum_mean_abs_feature_shap':float(np.abs(values[:,indices,j]).mean(0).sum())})
            feature_rows.extend(local_features);group_rows.extend(local_groups)
            diagnostics.append({**common,'pairs':PAIRS,'n_test':len(test),'n_features':len(cols),
                'max_additivity_error':error,'mean_mc_se':float(se.mean()),'max_mc_se':float(se.max())})
            status.append({**common,'status':'completed','error_type':''})
            if RUN_EFFECT_PLOTS:
                numeric=[f for f in cols if train[f].nunique(dropna=True)>1]
                corr=train[numeric].corr(method='spearman').abs()
                writable=corr.to_numpy(copy=True);np.fill_diagonal(writable,np.nan)
                maxcorr=pd.DataFrame(writable,index=corr.index,columns=corr.columns).max().fillna(0)
                prominence=pd.Series(np.abs(values).mean(axis=(0,2)),index=cols).sort_values(ascending=False)
                for method,candidates in [('PDP',[f for f in prominence.index if f in maxcorr and maxcorr[f]<=LOW_CORRELATION]),
                                           ('ALE',[f for f in prominence.index if f in maxcorr and maxcorr[f]>=HIGH_CORRELATION])]:
                    for feature in candidates[:EFFECT_TOP_N]:
                        edges=np.unique(train[feature].dropna().quantile(np.linspace(0,1,ALE_BINS+1)))
                        if method=='ALE':
                            if len(edges)<3:continue
                            result=ale(predict,X,feature,edges)
                            if result is None:continue
                            grid,effect,counts=result
                        else:
                            grid=np.unique(train[feature].dropna().quantile(np.linspace(.05,.95,PDP_POINTS)))
                            if len(grid)<2:continue
                            output=[]
                            for value in grid:
                                perturbed=X.copy();perturbed[feature]=value;output.append(predict(perturbed).mean(0))
                            effect=np.asarray(output)
                        fig,ax=plt.subplots(figsize=(8,5))
                        for j,k in enumerate(r.classes):ax.plot(grid,effect[:,j],label=class_name(r.task,k))
                        ax.set(title=f'{slug} · {method}',xlabel=pretty(feature),ylabel='Centered probability effect' if method=='ALE' else 'Mean probability')
                        ax.legend();fig.tight_layout();show(fig,folder/f'{method.lower()}_{safe(feature)}.png')
            print(f'{slug}: completed',flush=True)
        except Exception as exc:
            known=['OOF windows mismatch.','OOF labels mismatch.','Model does not reproduce saved OOF probabilities.',
                   'Invalid LOSO train/test provenance.','Class order mismatch.','Composite class order mismatch.',
                   'Shapley additivity check failed.']
            reason=str(exc) if str(exc) in known else type(exc).__name__
            status.append({**common,'status':'failed' if not any(s['experiment']==r.experiment and s['subject_id']==r.subject_id and s['status']=='completed' for s in status) else 'effect_plot_failed','error_type':reason})
            print(f'{slug}: {reason}',flush=True)
        pd.DataFrame(status).to_csv(OUT/'execution_status.csv',index=False)
        pd.DataFrame(feature_rows).to_csv(OUT/'shap_feature_importance_by_fold.csv',index=False)
        pd.DataFrame(group_rows).to_csv(OUT/'shap_group_importance.csv',index=False)
pd.DataFrame([{'category':w.category.__name__} for w in observed_warnings],columns=['category']).drop_duplicates().to_csv(OUT/'warning_categories.csv',index=False)
pd.DataFrame(diagnostics).to_csv(OUT/'sampling_diagnostics.csv',index=False)
display(pd.DataFrame(status))


## Summary, beeswarm, waterfall and stability — from saved results

Colors are percentiles of raw feature values within each test participant; missing values are gray. These model-agnostic attributions are defined on original input columns, unlike the previous transformed-tree version. Beeswarms use an equal number of sampled windows per available participant. Waterfalls default to S2, first saved window, and show every output class. Other-feature signed sums preserve additivity.

In [ ]:
features=pd.read_csv(OUT/'shap_feature_importance_by_fold.csv')
groups=pd.read_csv(OUT/'shap_group_importance.csv')
execution=pd.read_csv(OUT/'execution_status.csv')
success=execution[execution.status.eq('completed')][['experiment','subject_id']].drop_duplicates()
stability=[];variant_comparison=[]
for experiment,g in features.groupby('experiment'):
    per_subject=g.groupby(['subject_id','feature']).mean_abs_shap.mean().unstack(fill_value=0)
    for a,b in itertools.combinations(per_subject.index,2):
        x,y=per_subject.loc[a],per_subject.loc[b]
        topx=set(x.nlargest(min(10,len(x))).index);topy=set(y.nlargest(min(10,len(y))).index)
        stability.append(dict(experiment=experiment,subject_a=a,subject_b=b,
            rank_spearman=float(spearmanr(x,y).statistic) if x.nunique()>1 and y.nunique()>1 else np.nan,
            top10_jaccard=len(topx&topy)/len(topx|topy)))
    importance=per_subject.mean().sort_values(ascending=False).head(TOP_N).sort_values()
    fig,ax=plt.subplots(figsize=(11,max(5,len(importance)*.35)))
    ax.barh([pretty(x) for x in importance.index],importance.values)
    ax.set(title=experiment,xlabel='Equal-participant, equal-class mean |Shapley| (probability)')
    fig.tight_layout();show(fig,OUT/'summary'/f'shap_summary_{safe(experiment)}.png')
    entries=registry[registry.experiment.eq(experiment)]
    saved=[]
    for r in entries.itertuples():
        if not ((success.experiment==experiment)&(success.subject_id==r.subject_id)).any():continue
        folder=OUT/'folds'/f'{experiment}_S{r.subject_id}'
        table=pd.read_parquet(folder/f'shap_values_fold_{r.subject_id}.parquet')
        # Revalidate provenance/probabilities; prepare() never fits.
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            predict,train,test,cols,bg=prepare(r)
        raw=test.set_index('window_id')[cols]
        colors=raw.rank(pct=True)
        saved.append((r,table,cols,colors))
    if not saved:continue
    for k in saved[0][0].classes:
        count=min(BEESWARM_WINDOWS_PER_SUBJECT,min(len(t[t.class_label.eq(k)]) for _,t,_,_ in saved))
        parts=[];colorparts=[]
        for r,t,cols,colors in saved:
            t=t[t.class_label.eq(k)]
            sampled=t.sample(count,random_state=SEED+r.subject_id)
            parts.append(sampled[cols].reset_index(drop=True))
            colorparts.append(colors.loc[sampled.window_id,cols].reset_index(drop=True))
            if WATERFALL_SUBJECTS is None or r.subject_id in WATERFALL_SUBJECTS:
                row=t.iloc[0] if WATERFALL_WINDOW_ID is None else t[t.window_id.eq(WATERFALL_WINDOW_ID)].iloc[0]
                v=row[cols].astype(float);top=list(v.abs().nlargest(TOP_N).index)
                names=[pretty(f) for f in top];increments=list(v[top]);rest=[f for f in cols if f not in top]
                if rest:names.append(f'Other {len(rest)} features');increments.append(float(v[rest].sum()))
                current=float(row.base_value);final=current+float(v.sum())
                fig,ax=plt.subplots(figsize=(12,max(5,len(names)*.35)))
                for i,(name,increment) in enumerate(zip(names,increments)):
                    end=current+increment
                    ax.barh(i,abs(increment),left=min(current,end),color='firebrick' if increment>=0 else 'steelblue')
                    ax.text((current+end)/2,i,f'{increment:+.3g}',va='center',ha='center',fontsize=8);current=end
                ax.axvline(float(row.base_value),color='gray',linestyle='--',label=f'Reference {float(row.base_value):.3f}')
                ax.axvline(final,color='black',label=f'Output {final:.3f}')
                ax.set(yticks=range(len(names)),yticklabels=names,title=f'{experiment} · S{r.subject_id} · {class_name(r.task,k)}',xlabel='Predicted probability')
                ax.invert_yaxis();ax.legend();fig.tight_layout()
                show(fig,OUT/'waterfalls'/f'{safe(experiment)}_S{r.subject_id}_class{k}.png')
        all_values=pd.concat(parts,ignore_index=True).fillna(0);all_colors=pd.concat(colorparts,ignore_index=True).reindex(columns=all_values.columns)
        top=all_values.abs().mean().nlargest(TOP_N).index[::-1]
        fig,ax=plt.subplots(figsize=(12,max(5,len(top)*.4)));rng=np.random.default_rng(SEED)
        for i,f in enumerate(top):
            values=all_values[f].to_numpy();color=all_colors[f].to_numpy();finite=np.isfinite(color)
            jitter=rng.uniform(-.28,.28,len(values))
            ax.scatter(values[finite],i+jitter[finite],c=color[finite],cmap='coolwarm',vmin=0,vmax=1,s=10,alpha=.6)
            if (~finite).any():ax.scatter(values[~finite],i+jitter[~finite],color='gray',s=10,alpha=.5)
        ax.axvline(0,color='black',linewidth=.8)
        ax.set(yticks=range(len(top)),yticklabels=[pretty(f) for f in top],title=f'{experiment} · {class_name(saved[0][0].task,k)}',xlabel='Approximate Shapley contribution (probability)')
        fig.colorbar(ScalarMappable(norm=Normalize(0,1),cmap='coolwarm'),ax=ax,label='Raw feature percentile within test participant')
        fig.tight_layout();show(fig,OUT/'beeswarm'/f'{safe(experiment)}_class{k}.png')

stability=pd.DataFrame(stability);stability.to_csv(OUT/'shap_rank_stability.csv',index=False)
# Compare variants only within the same task, site and algorithm, on common participants.
for (task,site,model),g in features.groupby(['task','site','model']):
    full=g[g.variant.eq('full') & ~g.experiment.str.startswith('sensitivity_')]
    for variant in [v for v in g.variant.unique() if v!='full']:
        reduced=g[g.variant.eq(variant)]
        for sid in sorted(set(full.subject_id)&set(reduced.subject_id)):
            x=full[full.subject_id.eq(sid)].groupby('feature').mean_abs_shap.mean()
            y=reduced[reduced.subject_id.eq(sid)].groupby('feature').mean_abs_shap.mean()
            union=x.index.union(y.index);x=x.reindex(union,fill_value=0);y=y.reindex(union,fill_value=0)
            variant_comparison.append(dict(task=task,site=site,model=model,variant=variant,subject_id=sid,
                rank_spearman=float(spearmanr(x,y).statistic) if x.nunique()>1 and y.nunique()>1 else np.nan))
pd.DataFrame(variant_comparison).to_csv(OUT/'full_reduced_comparison.csv',index=False)
display(stability.groupby('experiment')[['rank_spearman','top10_jaccard']].mean() if not stability.empty else stability)


In [6]:
def md_table(frame):
    if frame.empty:return 'No results available.'
    return '\n'.join(['| '+' | '.join(frame.columns)+' |','| '+' | '.join(['---']*len(frame.columns))+' |']+
        ['| '+' | '.join(str(v).replace('|','/') for v in row)+' |' for row in frame.itertuples(index=False,name=None)])
report=['# All LOSO experiments — interpretability','',
    'Model-agnostic Monte Carlo Shapley values on class probabilities. No SHAP/Numba import and no training.',
    '## Available and executed experiments',md_table(execution.groupby(['experiment','status']).size().rename('folds').reset_index()),
    '## Unavailable experiments',md_table(pd.DataFrame(missing)),
    '## Group attributions',md_table(groups.groupby(['experiment','grouping','group'])[['mean_abs_group_sum','sum_mean_abs_feature_shap']].mean().reset_index()),
    '## Interpretation',
    'Attributions describe model predictions relative to sampled training reference rows; they do not identify causes of stress. '
    'All algorithms explain probabilities, but different tasks/classes/backgrounds still define different games. '
    'Fused identifies a model configuration: chest/wrist remain separate input groups. '
    'Family sum of signed contributions and sum of individual absolute importances differ because of cancellation.',
    '## Monte Carlo and dependencies',
    'Standard errors reflect sampling error conditional on the model, not uncertainty across people. '
    'Additivity does not establish convergence. Compare sampling budgets for sensitivity. '
    'Correlated-input hybrids may be out of support; ALE is first-order and descriptive. '
    'LOSO folds share training participants; rank stability is descriptive.',
    '## Availability limitations',
    'Unsaved models and missing provenance cannot be recovered from prediction CSVs alone. '
    'The calibration workflow must persist its complete predictor in a separate training run before it can be explained. '
    'No historical or synthetic artifact roots are combined with revision_v2.',
]
(OUT/'interpretability_report.md').write_text('\n\n'.join(report)+'\n',encoding='utf-8')
(OUT/'manifest.json').write_text(json.dumps(dict(method='antithetic_monte_carlo_shapley',pairs=PAIRS,
    output_scale='class_probability',artifact_root=ARTIFACTS_REL,no_training=True,no_numba=True,seed=SEED),indent=2),encoding='utf-8')
print('All-experiment report and figures saved successfully. No private absolute paths exported.')


All-experiment report and figures saved successfully. No private absolute paths exported.


## Interpretation of the all-experiment Shapley analysis

### Scope and execution coverage

The saved execution log contains **1,485 completed fold analyses across
99 experiment entries**, covering 15 held-out participants per entry.

These include primary classifiers, secondary binary tasks, advanced
algorithms, fusion methods, missing-modality scenarios, four-class models,
and complete/reduced sensitivity configurations.

The 99 entries include complete-model reruns from the sensitivity
workflow. They should not be interpreted as 99 independent model designs:
the separate benchmark identifies **84 distinct configurations after
recognizing 15 identical OOF reruns**.

The calibrated uncertainty experiment remains unavailable because its
complete fitted predictors and calibrators were not persisted.
Predictions alone cannot reconstruct that prediction function.

No model is trained or selected in this notebook.

### Attribution method and output scale

The analysis uses **Monte Carlo estimates of Shapley values**, based on
32 antithetic permutation pairs per fold.

The implementation does not import SHAP or Numba and is not exact TreeSHAP.
For every algorithm, it explains **class probabilities**.

Only held-out-participant windows receive explanations. Reference samples
come from training participants. Before explanation, the loaded predictor
must reproduce its saved out-of-fold probabilities.

Mean absolute contributions describe the magnitude of model reliance
relative to the chosen reference distribution. They do not indicate
whether increasing a feature increases stress probability; direction
requires class-specific signed explanations or effect plots.

### Numerical checks and approximation uncertainty

Across the completed folds, the largest additivity discrepancy is
approximately **2.1 × 10⁻¹⁵**.

This confirms that the estimated contributions and their sampled reference
reconstruct the model output to numerical precision.

However, additivity is a property of the sampling construction and
**does not establish convergence of individual feature attributions**.

The saved Monte Carlo diagnostics show:

| Diagnostic across fold analyses | Value |
|---|---:|
| Mean of fold-level mean standard errors | 0.0083 |
| Largest fold-level mean standard error | 0.0300 |
| Largest individual attribution standard error | 0.0913 |

These values are in probability units. For example, 0.0083 corresponds
to approximately 0.83 percentage points.

Some individual explanations therefore retain appreciable sampling
uncertainty. Closely ranked features should not be interpreted as
reliably ordered without checking a larger sampling budget.

These standard errors measure Monte Carlo approximation uncertainty,
not uncertainty across participants or confidence in a prediction.

The majority classifiers have **zero feature contributions**, consistent
with their input-independent predictions.

### Main three-class wrist model

For the original wrist random forest, the largest mean absolute
feature contributions include:

| Feature | Mean absolute contribution |
|---|---:|
| Wrist temperature mean | 0.0401 |
| Wrist EDA standard deviation | 0.0303 |
| Wrist ACC movement variance | 0.0282 |
| Wrist mean pulse rate | 0.0277 |
| Wrist ACC activity level | 0.0264 |

These values average across held-out participants and output classes.

At physiological-family level, electrodermal, pulse-related, movement,
and temperature inputs all contribute. The model does not rely
exclusively on one physiological system.

Temperature and movement contributions should not automatically be
interpreted as stress-specific physiology. Recording conditions, posture,
activity, and experimental-task characteristics are alternative
explanations requiring separate investigation.

### Secondary binary models

In both wrist logistic-regression binary tasks, pulse-related variables
are prominent.

For stress versus non-stress, the leading features include:

- Mean pulse rate: **0.0955** mean absolute contribution.
- PRV SDNN: **0.0912**.
- ACC movement variance: **0.0736**.
- BVP beat count: **0.0673**.
- Temperature mean: **0.0602**.

The BVP/PRV family also has the largest mean absolute grouped contribution
in both binary tasks.

This describes how these classifiers distinguish their labels.
It does not establish that pulse variability measures subjective stress
intensity, nor that PRV and ECG-derived HRV are interchangeable.

Attribution magnitudes should not be ranked directly against the
three-class task: the output classes, fitted functions, and reference
distributions differ.

### Late fusion and modality contributions

The late-fusion model combines separate chest and wrist logistic-regression
predictions using fixed equal weights.

Prominent individual features include wrist activity, wrist PRV SDNN,
chest posture, wrist pulse rate, and chest ECG SDNN.

Mean absolute signed-sum contributions by modality are:

| Model | Chest contribution | Wrist contribution |
|---|---:|---:|
| Early-fusion logistic regression | 0.2108 | 0.1521 |
| Late fusion | 0.1537 | 0.1454 |
| Early-fusion RF, filter 0.90 | 0.1011 | 0.0932 |

Late fusion has similar attribution magnitudes for its two input
modalities. This does not mean that every participant or prediction
depends equally on both.

A larger contribution is not evidence that removing that modality would
produce a proportional performance loss. Sensor utility requires the
separate predictive comparisons and missing-modality experiments.

### Reduced early-fusion random forest

For the random forest with correlation threshold 0.90, the leading
features include:

| Feature | Mean absolute contribution |
|---|---:|
| Chest ACC activity level | 0.0455 |
| Chest ECG beat count | 0.0342 |
| Wrist ACC movement variance | 0.0228 |
| Wrist EDA mean | 0.0214 |
| Wrist EDA standard deviation | 0.0210 |

Movement has the largest mean absolute grouped contribution in this
configuration, followed by electrodermal inputs.

This is compatible with a model using both physiological and
behavioral/contextual information. It does not demonstrate that movement
causes stress or that the classifier has isolated a stress-specific
mechanism.

Beat count is also related to the number of detected beats within a fixed
window and can reflect both rate and detection behavior. Its importance
should be interpreted alongside beat quality and raw-signal inspection.

### Wrist XGBoost with filter 0.95

The reduced wrist XGBoost configuration emphasizes temperature mean,
pulse rate, EDA mean, EDA variability, and an acceleration posture feature.

Its largest individual mean absolute contribution is temperature mean
(**0.1184**). Temperature also has the largest grouped contribution
among physiological families in this configuration.

This differs from the original wrist random forest and illustrates
that models with comparable predictive scores can use inputs differently.

A large attribution does not validate the feature as a physiological
biomarker. Correlation, feature representation, model structure, and
background choice all affect attribution.

### Four-class interpretation

For early-fusion random forest with meditation included, respiratory rate
has the largest observed mean absolute feature contribution
(**0.0409**), followed by chest activity and EMG variables.

Grouped chest contribution is larger than grouped wrist contribution
in this configuration.

These findings concern a different prediction task. They should not be
used to claim that respiration is universally the most important feature
for the primary three-class problem.

### Stability across held-out participants

Selected mean pairwise fold-stability values are:

| Configuration | Rank Spearman | Top-10 Jaccard |
|---|---:|---:|
| Original wrist random forest | 0.734 | 0.689 |
| Early-fusion logistic regression | 0.613 | 0.280 |
| Late fusion | 0.660 | 0.281 |
| Early-fusion RF, filter 0.90 | 0.935 | 0.570 |
| Wrist XGBoost, filter 0.95 | 0.833 | 0.705 |
| Four-class early-fusion RF | 0.893 | 0.630 |

Rank Spearman describes agreement across the full importance ranking.
Top-10 Jaccard describes overlap among the ten highest-ranked features.
They capture different aspects of stability.

For reduced models, excluded inputs receive zero attribution.
Shared zero-valued features and ties can increase apparent full-ranking
agreement. The high Spearman value of the reduced early-fusion random
forest should therefore be considered alongside its more moderate
top-10 overlap.

These are descriptive measures. Folds share training participants,
and Monte Carlo noise can contribute to ranking differences.

### Complete versus reduced models

The mean within-participant rank correlation between complete and reduced
feature attributions is approximately:

- **0.519** for early-fusion RF at threshold 0.90.
- **0.972** for wrist XGBoost at threshold 0.95.

Filtering can therefore substantially reorganize attribution in some
configurations while leaving rankings relatively similar in others.

The comparison includes zero contributions for removed features.
It does not establish that changes in predictive performance are caused
by changes in a particular feature's attribution.

### Reading the plots

- **Summary bars:** average magnitude of contributions, without direction.
- **Beeswarm plots:** distributions of signed contributions for a specified
  output class; colors are raw-feature percentiles within each participant.
- **Waterfalls:** an individual window's reference output plus feature
  contributions reconstructs its predicted class probability.
- **PDP:** model responses under feature perturbation, restricted here
  using low pairwise training correlation.
- **ALE:** centered local probability effects for highly correlated
  features, using training-defined bins and held-out observations.

Low pairwise correlation does not guarantee independence.
ALE reduces some extrapolation problems but is not causal and may be
unavailable when test bins lack observations.

### Group attribution and cancellation

Two group summaries are saved:

1. Mean absolute value of the signed sum of contributions within a group.
2. Sum of the individual mean absolute feature contributions.

They are not equivalent. Positive and negative contributions can cancel
within a physiological family or modality.

Neither summary should be described as a percentage of predictive
accuracy explained by that group.

### Overall conclusion

The explanations show that model behavior depends on the task, algorithm,
sensor configuration, and redundancy filter. Pulse-related features are
prominent in the wrist binary classifiers, while temperature, movement,
EDA, respiration, and muscle variables contribute differently across
three- and four-class models.

The analysis supports inspection of model reliance and individual
predictions. It does not identify causal mechanisms, validate physiological
biomarkers, or establish a predictive winner.

Important limitations remain: Monte Carlo approximation error, correlated
inputs, context-related features, shared LOSO training data, and the small
participant sample. Interpretation should therefore accompany not replace
the separate performance, robustness, and reliability analyses.